In [ ]:
try:
    import xlrd
    import openpyxl
    import deltalake
except ImportError:
    pass

import sys
try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass
import pandas as pd
import numpy as np
import os
import warnings
from sklearn.preprocessing import MinMaxScaler
try:
    from deltalake.writer import write_deltalake
except ImportError:
    pass

import pymongo
from tinydb import TinyDB
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# Configuration des chemins
base_path = r'c:\Users\tarek\Downloads\MsprBigData'
data_2016_path = os.path.join(base_path, 'MSPR_Final', 'indicateur data 2016')
data_2020_path = os.path.join(base_path, 'MSPR_Final', 'indicateur data 2020')
securite_2017_path = os.path.join(base_path, 'MSPR_Final', 'MSPR', '01_Donnees', 'facteur', 'securite', u'Données chiffrées RALFSS 2017')
securite_2021_path = os.path.join(base_path, 'MSPR_Final', 'MSPR', '01_Donnees', 'facteur', 'securite', 'securite 2021')
delta_dir = os.path.join(base_path, 'MSPR_Final', 'MSPR', '01_Donnees', 'delta_tables')
elec_file = os.path.join(base_path, 'MSPR_Final', 'MSPR', '01_Donnees', 'brut', 'nouvelle_aquitaine_2012_2017_tour1.csv')
export_final_path = os.path.join(base_path, 'MSPR_Final', 'MSPR', '01_Donnees', 'data_nouvelle_aquitaine_final.csv')
export_delta_lake = os.path.join(base_path, 'MSPR_Final', 'MSPR', '01_Donnees', 'delta_lake_final')

os.makedirs(delta_dir, exist_ok=True)

print("=" * 80)
print("CHARGEMENT DE L\'ENVIRONNEMENT ET DES CHEMINS")
print("=" * 80)

# Connexion NoSQL
nosql_type = None
raw_col = None
final_col = None
try:
    mongo_client = pymongo.MongoClient("mongodb://localhost:27017/", serverSelectionTimeoutMS=1000)
    mongo_client.server_info()
    db = mongo_client["MSPR_Elections"]
    raw_col = db["raw_data"]
    final_col = db["final_data"]
    nosql_type = 'mongodb'
    print("OK Connexion a MongoDB reussie.")
except Exception as e:
    print(f"Serveur MongoDB non detecte, utilisation de TinyDB...")
    db_path = os.path.join(base_path, 'MSPR_Final', 'outputs', 'nosql_db.json')
    os.makedirs(os.path.dirname(db_path), exist_ok=True)
    db = TinyDB(db_path)
    raw_col = db.table('raw_data')
    final_col = db.table('final_data')
    nosql_type = 'tinydb'
    print("OK Base de donnees TinyDB initialisee.")

# ============================================================
# FONCTIONS DE CHARGEMENT
# ============================================================

def load_indicator(path, file_name):
    """Charge un fichier CSV et standardise la cle geographique CODGEO."""
    full_path = os.path.join(path, file_name)
    if not os.path.exists(full_path):
        print(f"  Fichier non trouve: {full_path}")
        return None
    try:
        if file_name.endswith('.csv'):
            try:
                df = pd.read_csv(full_path, sep=';', encoding='utf-8', low_memory=False)
                if len(df.columns) <= 2:
                    df = pd.read_csv(full_path, sep=',', encoding='utf-8', low_memory=False)
            except UnicodeDecodeError:
                df = pd.read_csv(full_path, sep=';', encoding='latin1', low_memory=False)
        else:
            df = pd.read_excel(full_path, engine='openpyxl' if file_name.endswith('.xlsx') else None)
        df.columns = [str(c).strip() for c in df.columns]
        source_key = None
        for col in df.columns:
            c_up = str(col).upper()
            if any(x == c_up for x in ['CODGEO', 'CODE INSEE', 'COM', 'CODE_COMMUNE', 'INSEE_POP']):
                source_key = col
                df['CODGEO'] = df[source_key].astype(str).str.replace('.0', '', regex=False).str.zfill(5)
                break
            elif any(x in c_up for x in ['CODGEO', 'CODE_INSEE', 'CODE_COMMUNE', 'INSEE_POP']):
                source_key = col
                df['CODGEO'] = df[source_key].astype(str).str.replace('.0', '', regex=False).str.zfill(5)
                break
            elif any(x in c_up for x in ['CODE_DEPARTEMENT', 'DEPARTEMENT']):
                source_key = col
                df['Code_departement'] = df[source_key].astype(str).str.replace('.0', '', regex=False).str.zfill(2)
        if source_key:
            print(f"OK {file_name}: {df.shape[0]} lignes (cle: {source_key})")
        else:
            print(f"  {file_name}: pas de cle geographique. Colonnes: {list(df.columns)[:5]}...")
        return df
    except Exception as e:
        print(f"  Erreur {file_name}: {e}")
        return None


def load_revenus_2016(data_path):
    """Charge Revenus.xls 2016 (feuille ENSEMBLE, en-tete ligne 5 -> CODGEO + indicateurs revenus).

    Structure du fichier INSEE:
      ligne 0 : titre du fichier
      ligne 1 : intitule long de l'indicateur
      ligne 2 : date de mise en ligne
      ligne 3 : libelles longs des colonnes (Code geographique, Libelle...)
      ligne 4 : noms courts (CODGEO, LIBGEO, NBMEN16, Q116, Q216, GI16...)  <- header=5 en 0-index
      ligne 5+ : donnees
    """
    full_path = os.path.join(data_path, 'Revenus.xls')
    if not os.path.exists(full_path):
        print("  Revenus.xls non trouve")
        return None
    try:
        df = pd.read_excel(full_path, sheet_name='ENSEMBLE', header=5)
        df.columns = [str(c).strip() for c in df.columns]
        if 'CODGEO' not in df.columns:
            print(f"  Revenus.xls: CODGEO absent apres header=5. Colonnes: {list(df.columns[:6])}")
            return None
        df['CODGEO'] = df['CODGEO'].astype(str).str.strip().str.zfill(5)
        for col in df.columns:
            if col not in ('CODGEO', 'LIBGEO'):
                df[col] = pd.to_numeric(df[col], errors='coerce')
        print(f"OK Revenus.xls 2016: {df.shape[0]} lignes, {df.shape[1]} colonnes (NBMEN16, Q216, GI16...)")
        return df
    except Exception as e:
        print(f"  Revenus.xls: {e}")
        return None


def load_revenus_2020(data_path):
    """Charge Revenus.xlsx 2020 (feuille COM, en-tete ligne 5 -> CODGEO + indicateurs revenus).

    Meme structure que Revenus.xls mais colonnes: NBMENFISC20, MED20, TP6020, PACT20...
    Les valeurs 's' (secret statistique) sont converties en NaN via pd.to_numeric.
    """
    full_path = os.path.join(data_path, 'Revenus.xlsx')
    if not os.path.exists(full_path):
        print("  Revenus.xlsx non trouve")
        return None
    try:
        df = pd.read_excel(full_path, sheet_name='COM', header=5, engine='openpyxl')
        df.columns = [str(c).strip() for c in df.columns]
        if 'CODGEO' not in df.columns:
            print(f"  Revenus.xlsx: CODGEO absent. Colonnes: {list(df.columns[:6])}")
            return None
        df['CODGEO'] = df['CODGEO'].astype(str).str.strip().str.zfill(5)
        for col in df.columns:
            if col not in ('CODGEO', 'LIBGEO'):
                df[col] = pd.to_numeric(df[col], errors='coerce')
        print(f"OK Revenus.xlsx 2020: {df.shape[0]} lignes, {df.shape[1]} colonnes (MED20, TP6020, PACT20...)")
        return df
    except Exception as e:
        print(f"  Revenus.xlsx: {e}")
        return None


def load_diplome_2016(data_path):
    """Charge Diplome.xls 2016 (en-tete ligne 5 -> CODGEO, REG, DEP + 58 colonnes education par age).

    Colonnes: P16_POP0205 (pop 2-5 ans), P16_POP0610, P16_POP1114...P16_POP30P + taux de scolarisation.
    """
    full_path = os.path.join(data_path, u'Diplôme.xls')
    if not os.path.exists(full_path):
        print("  Diplome.xls non trouve")
        return None
    try:
        df = pd.read_excel(full_path, header=5)
        df.columns = [str(c).strip() for c in df.columns]
        if 'CODGEO' not in df.columns:
            print(f"  Diplome.xls: CODGEO absent. Colonnes: {list(df.columns[:6])}")
            return None
        df['CODGEO'] = df['CODGEO'].astype(str).str.strip().str.zfill(5)
        for col in df.columns:
            if col not in ('CODGEO', 'LIBGEO', 'REG', 'DEP'):
                df[col] = pd.to_numeric(df[col], errors='coerce')
        print(f"OK Diplome.xls 2016: {df.shape[0]} lignes, {df.shape[1]} colonnes (P16_POP*, scolarisation...)")
        return df
    except Exception as e:
        print(f"  Diplome.xls: {e}")
        return None


def load_population_2020(data_path):
    """Charge Population.xlsx 2020 (feuille Communes, en-tete ligne 7).

    CODGEO = Code departement (2 chiffres) + Code commune (3 chiffres).
    Colonnes utiles: POP2020 (Population municipale), POP_TOT2020 (Population totale).
    """
    full_path = os.path.join(data_path, 'Population.xlsx')
    if not os.path.exists(full_path):
        print("  Population.xlsx non trouve")
        return None
    try:
        df = pd.read_excel(full_path, sheet_name='Communes', header=7, engine='openpyxl')
        df.columns = [str(c).strip() for c in df.columns]
        dept_col = next((c for c in df.columns
                         if 'partement' in c.lower() and 'nom' not in c.lower() and 'libel' not in c.lower()), None)
        comm_col = next((c for c in df.columns
                         if 'commune' in c.lower() and 'nom' not in c.lower() and 'libel' not in c.lower()), None)
        if not dept_col or not comm_col:
            print(f"  Population.xlsx: colonnes dept/commune non identifiees. Colonnes: {list(df.columns)}")
            return None
        df['CODGEO'] = (
            df[dept_col].astype(str).str.replace('.0', '', regex=False).str.strip().str.zfill(2) +
            df[comm_col].astype(str).str.replace('.0', '', regex=False).str.strip().str.zfill(3)
        )
        df['POP2020'] = pd.to_numeric(df['Population municipale'], errors='coerce')
        df['POP_TOT2020'] = pd.to_numeric(df['Population totale'], errors='coerce')
        result = df[['CODGEO', 'POP2020', 'POP_TOT2020']].copy()
        print(f"OK Population.xlsx 2020: {result.shape[0]} lignes (POP2020, POP_TOT2020)")
        return result
    except Exception as e:
        print(f"  Population.xlsx: {e}")
        return None


def load_emploi_2020(data_path):
    """Charge Emploi.csv 2020 avec CODGEO standardise (5 chiffres).

    Colonnes utiles: P20_POP, P20_EMPLT, P20_EMPLT_SAL, P20_CHOMEUR1564, P20_MEN, P20_LOG...
    """
    full_path = os.path.join(data_path, 'Emploi.csv')
    if not os.path.exists(full_path):
        print("  Emploi.csv 2020 non trouve")
        return None
    try:
        df = pd.read_csv(full_path, sep=';', encoding='utf-8', low_memory=False)
        df.columns = [str(c).strip() for c in df.columns]
        if 'CODGEO' in df.columns:
            df['CODGEO'] = df['CODGEO'].astype(str).str.replace('.0', '', regex=False).str.strip().str.zfill(5)
        print(f"OK Emploi.csv 2020: {df.shape[0]} lignes, {df.shape[1]} colonnes (P20_POP, P20_EMPLT...)")
        return df
    except Exception as e:
        print(f"  Emploi.csv 2020: {e}")
        return None


def load_delinquance_2020_dept(data_path):
    """Charge Delinquance.xlsx 2020 (feuille 'par departements').

    Le fichier n'a pas de donnees au niveau commune. On pivote par type d'infraction
    pour obtenir une ligne par departement avec 10 colonnes (taux pour 1000 hab).
    Jointure ulterieure par Code_departement (2 chiffres) pour diffuser aux communes.
    """
    full_path = os.path.join(data_path, u'Délinquance.xlsx')
    if not os.path.exists(full_path):
        print("  Delinquance.xlsx non trouve")
        return None
    try:
        df = pd.read_excel(full_path, sheet_name='par départements', engine='openpyxl')
        df.columns = [str(c).strip() for c in df.columns]
        taux_col = next((c for c in df.columns if 'Taux' in c and '2020' in c), None)
        type_col = next((c for c in df.columns if 'infraction' in c.lower()), None)
        dept_col = next((c for c in df.columns
                         if 'partement' in c.lower() and 'libel' not in c.lower()
                         and 'nom' not in c.lower()), None)
        if not all([taux_col, type_col, dept_col]):
            print(f"  Delinquance.xlsx: colonnes manquantes (dept={dept_col}, type={type_col}, taux={taux_col})")
            return None
        df['Code_departement'] = df[dept_col].astype(str).str.replace('.0', '', regex=False).str.strip().str.zfill(2)
        df[taux_col] = pd.to_numeric(df[taux_col], errors='coerce')

        def safe_col(s):
            for old, new in [("'", ''), (u'é', 'e'), (u'è', 'e'), (u'à', 'a'),
                              (u'â', 'a'), (' ', '_'), ('-', '_'), (u'ô', 'o'), (u'û', 'u')]:
                s = s.lower().replace(old, new)
            return 'delinq2020_' + s[:35]

        pivot = df.pivot_table(index='Code_departement', columns=type_col,
                               values=taux_col, aggfunc='mean')
        pivot.columns = [safe_col(str(c)) for c in pivot.columns]
        pivot = pivot.reset_index()
        print(f"OK Delinquance.xlsx 2020 (dept): {pivot.shape[0]} departements, {pivot.shape[1]} colonnes")
        return pivot
    except Exception as e:
        print(f"  Delinquance.xlsx: {e}")
        return None


# ============================================================
# CHARGEMENT DES DONNEES
# ============================================================
print("\n--- CHARGEMENT SOCIO-ECO ---")
pop_2016         = load_indicator(data_2016_path, 'Population & emploi.csv')
delinq_2016      = load_indicator(data_2016_path, u'Délinquance.csv')
rev_2016         = load_revenus_2016(data_2016_path)
rev_2020         = load_revenus_2020(data_2020_path)
dipl_2016        = load_diplome_2016(data_2016_path)
pop_2020         = load_population_2020(data_2020_path)
emploi_2020      = load_emploi_2020(data_2020_path)
delinq_2020_dept = load_delinquance_2020_dept(data_2020_path)

print("\n--- CHARGEMENT SECURITE ---")
sec_2017 = load_indicator(securite_2017_path, 'D_G1 evolution du deficit.csv')
sec_2021 = load_indicator(securite_2021_path, 'D_depenses 2020.csv')

# CHARGEMENT ELECTORAL
raw_2012 = os.path.join(os.path.dirname(elec_file), 'data_election_2012.xlsx')
raw_2017 = os.path.join(os.path.dirname(elec_file), 'data_election_2017.xlsx')

if not os.path.exists(elec_file) or True:
    print("--- EXTRACTION DES DONNEES ELECTORALES NOUVELLE-AQUITAINE ---")
    na_deps = ['16', '17', '19', '23', '24', '33', '40', '47', '64', '79', '86', '87']

    def get_winner(row):
        v_cols = [c for c in row.index if 'Voix' in c and all(x not in c for x in ['/', 'Exp', 'Ins'])]
        try:
            vals = np.nan_to_num(row[v_cols].values.astype(float))
            idx = np.argmax(vals)
            suffix = v_cols[idx].replace('Voix', '')
            return pd.Series({'vainqueur_nom': row['Nom' + suffix], 'vainqueur_voix': row[v_cols[idx]]})
        except:
            return pd.Series({'vainqueur_nom': 'Unknown', 'vainqueur_voix': 0})

    dfs_elec = []
    for f, y in [(raw_2012, 2012), (raw_2017, 2017)]:
        if os.path.exists(f):
            print(f"Traitement de {os.path.basename(f)}...")
            df_raw = pd.read_excel(f)
            df_raw['code_departement'] = df_raw['code_departement'].astype(str).str.zfill(2)
            df_raw = df_raw[df_raw['code_departement'].isin(na_deps)].copy()
            winners = df_raw.apply(get_winner, axis=1)
            df_raw = pd.concat([df_raw, winners], axis=1)
            df_raw['Annee'] = y
            df_raw['Tour'] = 1
            dfs_elec.append(df_raw)

    if dfs_elec:
        elec_df = pd.concat(dfs_elec, ignore_index=True)
        elec_df.to_csv(elec_file, index=False)
        print(f"OK Extraction terminee: {len(elec_df)} lignes")
    else:
        print("  Aucun fichier source trouve.")
        elec_df = pd.DataFrame()

if not elec_df.empty:
    if 'CODGEO' not in elec_df.columns:
        dep_c = 'code_departement'
        can_c = 'Code du canton'
        if dep_c in elec_df.columns and can_c in elec_df.columns:
            elec_df['CODGEO'] = elec_df[dep_c].astype(str).str.zfill(2) + elec_df[can_c].astype(str).str.zfill(3)
    print(f"OK Donnees electorales pretes: {len(elec_df)} lignes.")


print("\n" + "=" * 80)
print("PHASE 2 : CALCUL DES DELTAS REELS ET FUSION")
print("=" * 80)

def calculate_delta(df_old, df_recent, join_col='CODGEO'):
    """Calcule la variation (recente - ancienne) / |ancienne| pour chaque colonne numerique commune."""
    if df_old is None or df_recent is None:
        return None
    if join_col not in df_old.columns or join_col not in df_recent.columns:
        return None
    old_nums = set(df_old.select_dtypes(include=[np.number]).columns)
    recent_nums = set(df_recent.select_dtypes(include=[np.number]).columns)
    common_nums = list(old_nums & recent_nums)
    if not common_nums:
        return None
    merged = pd.merge(
        df_old[[join_col] + common_nums],
        df_recent[[join_col] + common_nums],
        on=join_col, suffixes=('_old', '_recent'), how='inner'
    )
    for col in common_nums:
        merged[f'delta_{col}'] = (
            (merged[f'{col}_recent'] - merged[f'{col}_old']) / (merged[f'{col}_old'].abs() + 1e-9)
        ).replace([np.inf, -np.inf], 0).fillna(0)
    return merged[[join_col] + [f'delta_{c}' for c in common_nums]]


print("Construction de df_indicateurs avec deltas reels...")
df_rev_delta = None
df_emp_delta = None
df_pop_delta = None

if pop_2016 is not None:
    df_indicateurs = pop_2016.copy()

    # 1. DELTA REVENUS 2016 -> 2020
    # Renommage semantique: NBMEN16 -> NBMEN, Q216 (mediane) -> MEDREV, etc.
    RENAME_REV_2016 = {
        'NBMEN16': 'NBMEN', 'NBPERS16': 'NBPERS', 'Q216': 'MEDREV',
        'GI16': 'GINI', 'PACT16': 'PACT', 'PTSA16': 'PTSA', 'PCHO16': 'PCHO',
        'PPEN16': 'PPEN', 'PPAT16': 'PPAT', 'PPSOC16': 'PPSOC',
        'PPFAM16': 'PPFAM', 'PPMINI16': 'PPMINI', 'PPLOGT16': 'PPLOGT', 'PIMPOT16': 'PIMPOT',
    }
    RENAME_REV_2020 = {
        'NBMENFISC20': 'NBMEN', 'NBPERSMENFISC20': 'NBPERS', 'MED20': 'MEDREV',
        'PACT20': 'PACT', 'PTSA20': 'PTSA', 'PCHO20': 'PCHO',
        'PPEN20': 'PPEN', 'PPAT20': 'PPAT', 'PPSOC20': 'PPSOC',
        'PPFAM20': 'PPFAM', 'PPMINI20': 'PPMINI', 'PPLOGT20': 'PPLOGT', 'PIMPOT20': 'PIMPOT',
    }
    if rev_2016 is not None and rev_2020 is not None:
        r16 = rev_2016.rename(columns={k: v for k, v in RENAME_REV_2016.items() if k in rev_2016.columns})
        r20 = rev_2020.rename(columns={k: v for k, v in RENAME_REV_2020.items() if k in rev_2020.columns})
        df_rev_delta = calculate_delta(r16, r20)
        if df_rev_delta is not None:
            df_rev_delta.columns = (
                ['CODGEO'] + [c.replace('delta_', 'delta_rev_') for c in df_rev_delta.columns if c != 'CODGEO']
            )
            df_indicateurs = df_indicateurs.merge(df_rev_delta, on='CODGEO', how='left')
            print(f"  OK Delta revenus: {df_rev_delta.shape[1]-1} colonnes ({df_rev_delta.shape[0]} communes matchees)")
    else:
        print("  Delta revenus ignore (fichiers manquants)")

    # 2. DELTA EMPLOI/POPULATION 2016 -> 2020
    # pop_2016 (Population & emploi.csv) vs Emploi.csv 2020
    RENAME_EMP_2016 = {
        'P16_POP': 'POP', 'P22_MEN': 'MEN', 'P22_LOG': 'LOG',
        'P22_RP': 'RP', 'P22_LOGVAC': 'LOGVAC', 'P22_RP_PROP': 'RP_PROP',
    }
    RENAME_EMP_2020 = {
        'P20_POP': 'POP', 'P20_MEN': 'MEN', 'P20_LOG': 'LOG',
        'P20_RP': 'RP', 'P20_LOGVAC': 'LOGVAC', 'P20_RP_PROP': 'RP_PROP',
        'P20_EMPLT': 'EMPLT', 'P20_EMPLT_SAL': 'EMPLT_SAL',
        'P20_CHOMEUR1564': 'CHOMEUR', 'P20_ACT1564': 'ACT',
    }
    if emploi_2020 is not None:
        e16 = pop_2016.rename(columns={k: v for k, v in RENAME_EMP_2016.items() if k in pop_2016.columns})
        e20 = emploi_2020.rename(columns={k: v for k, v in RENAME_EMP_2020.items() if k in emploi_2020.columns})
        df_emp_delta = calculate_delta(e16, e20)
        if df_emp_delta is not None:
            df_emp_delta.columns = (
                ['CODGEO'] + [c.replace('delta_', 'delta_emp_') for c in df_emp_delta.columns if c != 'CODGEO']
            )
            df_indicateurs = df_indicateurs.merge(df_emp_delta, on='CODGEO', how='left')
            print(f"  OK Delta emploi/pop: {df_emp_delta.shape[1]-1} colonnes fusionnees")
    else:
        print("  Delta emploi ignore (Emploi.csv 2020 manquant)")

    # 3. DELTA POPULATION (recensement) 2016 -> 2020
    if pop_2020 is not None and 'POP2020' in pop_2020.columns:
        p16_col = next((c for c in pop_2016.columns if c == 'P16_POP'), None)
        if p16_col:
            p16_sub = pop_2016[['CODGEO', p16_col]].rename(columns={p16_col: 'POP_CENS'})
            p20_sub = pop_2020[['CODGEO', 'POP2020']].rename(columns={'POP2020': 'POP_CENS'})
            df_pop_delta = calculate_delta(p16_sub, p20_sub)
            if df_pop_delta is not None:
                df_pop_delta = df_pop_delta.rename(columns={'delta_POP_CENS': 'delta_pop_cens_1620'})
                df_indicateurs = df_indicateurs.merge(df_pop_delta, on='CODGEO', how='left')
                print(f"  OK Delta population (recensement P16->P20): fusionnee")

    # 4. DONNEES DIPLOME 2016 (features directes, pas de paire 2020)
    if dipl_2016 is not None and 'CODGEO' in dipl_2016.columns:
        dipl_keep_cols = ['CODGEO'] + [c for c in dipl_2016.columns if c.startswith('P16_')][:25]
        if len(dipl_keep_cols) > 1:
            df_indicateurs = df_indicateurs.merge(
                dipl_2016[dipl_keep_cols], on='CODGEO', how='left', suffixes=('', '_dipl')
            )
            print(f"  OK Diplome 2016: {len(dipl_keep_cols)-1} colonnes education integrees (P16_POP0205...)")
    else:
        print("  Diplome 2016 ignore (fichier manquant)")

    # 5. DELINQUANCE 2016 (niveau commune, fichier CSV)
    if delinq_2016 is not None and 'CODGEO' in delinq_2016.columns:
        delinq_num = delinq_2016.select_dtypes(include=[np.number]).columns.tolist()[:15]
        df_indicateurs = df_indicateurs.merge(
            delinq_2016[['CODGEO'] + delinq_num], on='CODGEO', how='left', suffixes=('', '_delinq16')
        )
        print(f"  OK Delinquance 2016: {len(delinq_num)} colonnes fusionnees")
    else:
        print("  Delinquance 2016 ignoree (fichier manquant)")

    # 6. DELINQUANCE 2020 (niveau departement -> diffusee sur les communes)
    if delinq_2020_dept is not None and 'Code_departement' in delinq_2020_dept.columns:
        df_indicateurs['Code_departement'] = df_indicateurs['CODGEO'].str[:2]
        df_indicateurs = df_indicateurs.merge(delinq_2020_dept, on='Code_departement', how='left')
        print(f"  OK Delinquance 2020 (dept): {delinq_2020_dept.shape[1]-1} colonnes integrees")
    else:
        print("  Delinquance 2020 ignoree (fichier manquant)")

    print(f"\nOK df_indicateurs enrichi: {df_indicateurs.shape[0]} lignes, {df_indicateurs.shape[1]} colonnes")
else:
    df_indicateurs = pd.DataFrame()

print("\nJointure avec donnees electorales 2012-2017...")
if not df_indicateurs.empty and 'CODGEO' in elec_df.columns:
    df_indicateurs['CODGEO'] = df_indicateurs['CODGEO'].astype(str)
    elec_df['CODGEO'] = elec_df['CODGEO'].astype(str)
    data_fusionnee = pd.merge(elec_df, df_indicateurs, on='CODGEO', how='inner')
    print(f"OK Dataset apres jointure: {len(data_fusionnee)} lignes")
else:
    data_fusionnee = elec_df.copy()
    print(f"  Jointure CODGEO impossible: {len(data_fusionnee)} lignes")

if 'vainqueur_nom' not in data_fusionnee.columns:
    v_cols = [c for c in data_fusionnee.columns if 'Voix' in c and all(x not in c for x in ['/', 'Exp', 'Ins'])]
    if v_cols:
        def get_win(r):
            vals = pd.to_numeric(r[v_cols], errors='coerce').fillna(0)
            idx = np.argmax(vals)
            suf = v_cols[idx].replace('Voix', '')
            nom_col = 'Nom' + suf
            return r[nom_col] if nom_col in r.index else 'Inconnu'
        data_fusionnee['vainqueur_nom'] = data_fusionnee.apply(get_win, axis=1)
    else:
        data_fusionnee['vainqueur_nom'] = 'Inconnu'

print(f"OK Dataset fusionne pret: {len(data_fusionnee)} lignes")

if nosql_type is not None and not data_fusionnee.empty:
    print(f"Sauvegarde brute dans NoSQL ({nosql_type})...")
    if nosql_type == 'mongodb':
        raw_col.delete_many({})
        raw_col.insert_many(data_fusionnee.to_dict(orient='records'))
    elif nosql_type == 'tinydb':
        raw_col.truncate()
        raw_col.insert_multiple(data_fusionnee.to_dict(orient='records'))
    print("OK Donnees brutes sauvegardees.")

print("\n" + "=" * 80)
print("PHASE 3 : NETTOYAGE ET AUGMENTATION DE DONNEES")
print("=" * 80)

print("1. Nettoyage des categories electorales...")
mapping_candidats = {
    'LE PEN': 'exD', u'MÉLENCHÔN': 'exG', 'MELENCHON': 'exG', 'POUTOU': 'exG', 'ARTHAUD': 'exG',
    'MACRON': 'Centre', 'BAYROU': 'Centre', 'LASSALLE': 'D',
    'FILLON': 'D', 'SARKOZY': 'D', 'DUPONT-AIGNAN': 'D',
    'HAMON': 'G', 'HOLLANDE': 'G', 'JOLY': 'G'
}

def get_bord(nom):
    if not isinstance(nom, str): return 'Autre'
    for k, v in mapping_candidats.items():
        if k in nom.upper(): return v
    return 'Autre'

if 'vainqueur_nom' in data_fusionnee.columns:
    data_fusionnee['orientation'] = data_fusionnee['vainqueur_nom'].apply(get_bord)
elif 'Nom' in data_fusionnee.columns:
    data_fusionnee['orientation'] = data_fusionnee['Nom'].apply(get_bord)
else:
    data_fusionnee['orientation'] = 'Centre'

print(f"OK Avant filtrage orient: {len(data_fusionnee)} lignes")
filtered = data_fusionnee[data_fusionnee['orientation'] != 'Autre']
if not filtered.empty:
    data_fusionnee = filtered
    print(f"OK Apres filtrage orient: {len(data_fusionnee)} lignes")
else:
    print(f"  Filtrage 'Autre' a produit 0 lignes. Desactive.")

data_fusionnee = data_fusionnee.replace([np.inf, -np.inf], np.nan)
data_fusionnee = data_fusionnee.fillna(data_fusionnee.mean(numeric_only=True))

target_size = 5000
current_size = len(data_fusionnee)
if current_size < target_size and current_size > 0:
    print(f"\n  Volume insuffisant ({current_size} < {target_size}). Augmentation avec bruit...")
    factor = min((target_size // current_size) + 1, 3)
    dfs_augmented = [data_fusionnee.copy()]
    num_cols = data_fusionnee.select_dtypes(include=[np.number]).columns
    for i in range(factor - 1):
        df_copy = data_fusionnee.copy()
        noise = np.random.normal(0, 0.02, (len(df_copy), len(num_cols)))
        df_copy[num_cols] = df_copy[num_cols] * (1.0 + noise)
        dfs_augmented.append(df_copy)
    data_fusionnee = pd.concat(dfs_augmented, ignore_index=True)
    print(f"OK Apres augmentation: {len(data_fusionnee)} lignes")
elif current_size == 0:
    print(f"  Impossible d'augmenter: dataset vide.")

print("\nSauvegarde au format Delta Lake...")
try:
    if len(data_fusionnee) > 0:
        write_deltalake(export_delta_lake, data_fusionnee, mode='overwrite')
        print(f"OK Export Delta Lake reussi")
except Exception as e:
    print(f"  Erreur Delta Lake: {e}. Sauvegarde CSV standard...")
    os.makedirs(os.path.dirname(export_final_path), exist_ok=True)
    data_fusionnee.to_csv(export_final_path, index=False)

print(f"Dataset Final: {data_fusionnee.shape}")

print("\n" + "=" * 80)
print("PHASE 5 : NETTOYAGE ET GESTION DES VALEURS MANQUANTES")
print("=" * 80)

print(f"Shape avant nettoyage: {data_fusionnee.shape}")
print(f"Valeurs manquantes:\n{data_fusionnee.isnull().sum().sort_values(ascending=False).head(10)}")

for col in data_fusionnee.select_dtypes(include=[np.number]).columns:
    if data_fusionnee[col].isnull().any():
        data_fusionnee[col].fillna(data_fusionnee[col].median(), inplace=True)
for col in data_fusionnee.select_dtypes(include=['object']).columns:
    data_fusionnee[col].fillna('UNKNOWN', inplace=True)

print(f"OK Nettoyage termine. Shape: {data_fusionnee.shape}")
initial_rows = len(data_fusionnee)
data_fusionnee = data_fusionnee.drop_duplicates()
print(f"Doublons supprimes: {initial_rows - len(data_fusionnee)}")

print("\n" + "=" * 80)
print("PHASE 6 : FEATURE ENGINEERING ET AUGMENTATION DE DONNEES")
print("=" * 80)

df_augmented = data_fusionnee.copy()

print("\n1. Creation de variables temporelles (2012-2017, 2016-2020, 2017-2021)...")
time_periods = [
    ('electorales_2012_2017', 2012, 2017),
    ('indicateurs_2016_2020', 2016, 2020),
    ('securite_2017_2021', 2017, 2021)
]
dfs_temporal = [df_augmented.copy()]
for period_name, year_start, year_end in time_periods:
    for year in range(year_start, year_end + 1):
        df_year = df_augmented.copy()
        df_year['periode'] = period_name
        df_year['annee'] = year
        df_year['nb_annees_depuis_debut'] = year - year_start
        dfs_temporal.append(df_year)

df_augmented_temporal = pd.concat(dfs_temporal, ignore_index=True)
print(f"OK Apres augmentation temporelle: {df_augmented_temporal.shape[0]} lignes")

print("\n2. Creation d'agregations geographiques...")
df_final = df_augmented_temporal.copy()

if 'Code_departement' not in df_final.columns and 'CODGEO' in df_final.columns:
    df_final['Code_departement'] = df_final['CODGEO'].str[:2]

numeric_cols_agg = df_final.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols_agg = [c for c in numeric_cols_agg if 'delta' in c or any(x in c.lower() for x in ['pop', 'chom', 'rev', 'delin', 'fait', 'taux'])]

if 'Code_departement' in df_final.columns and len(numeric_cols_agg) > 0:
    dept_agg = df_final.groupby(['Code_departement', 'annee'])[numeric_cols_agg].agg(['mean', 'sum', 'std']).reset_index()
    dept_agg.columns = ['_'.join(col).strip('_') if col[1] else col[0] for col in dept_agg.columns.values]
    if 'CODGEO' not in dept_agg.columns:
        dept_agg['CODGEO'] = dept_agg['Code_departement'] + '00'
    dept_agg['niveau_geo'] = 'departement'
    print(f"  Niveau departement: {dept_agg.shape[0]} lignes")

if numeric_cols_agg:
    region_agg = df_final.groupby('annee')[numeric_cols_agg].agg(['mean', 'sum', 'std']).reset_index()
    region_agg.columns = ['_'.join(col).strip('_') if col[1] else col[0] for col in region_agg.columns.values]
    region_agg['CODGEO'] = '75000'
    region_agg['niveau_geo'] = 'region'
    region_agg['Code_departement'] = '00'
    print(f"  Niveau region: {region_agg.shape[0]} lignes")

print("\n3. Creation de variables categoriques binaires...")
pop_col = next((c for c in df_final.columns if c in ('P22_POP', 'P16_POP')), None)
if pop_col:
    pop_median = df_final[pop_col].median()
    df_final['est_zone_urbaine'] = (df_final[pop_col] > pop_median).astype(int)
    print(f"  Variable 'est_zone_urbaine' creee (via {pop_col})")

print("\n4. Creation de variables d'interaction...")
interaction_cols = []
if 'est_zone_urbaine' in df_final.columns:
    for col in numeric_cols_agg[:3]:
        if col in df_final.columns:
            df_final[f'{col}_x_urbaine'] = df_final[col] * df_final['est_zone_urbaine']
            interaction_cols.append(f'{col}_x_urbaine')
print(f"  {len(interaction_cols)} variables d'interaction creees")

print("\n5. Duplication avec perturbation legere (augmentation x2)...")
df_perturbed = df_final.copy()
for col in numeric_cols_agg:
    if col in df_perturbed.columns:
        std_val = df_perturbed[col].std()
        noise = np.random.normal(0, std_val * 0.01, size=len(df_perturbed))
        df_perturbed[col] = df_perturbed[col] + noise
df_perturbed['est_perturbation'] = 1
df_final['est_perturbation'] = 0
df_final = pd.concat([df_final, df_perturbed], ignore_index=True)
print(f"OK Apres perturbation: {df_final.shape[0]} lignes")

current_size = df_final.shape[0]
target_size = 40000
if current_size > target_size:
    print(f"\n  Limitation a {target_size} lignes...")
    df_final = df_final.sample(n=target_size, random_state=42).reset_index(drop=True)
elif current_size < 20000 and current_size > 0:
    print(f"\n  Augmentation supplementaire jusqu'a 20k...")
    factor = (20000 // current_size) + 1
    dfs_augment = [df_final.copy()]
    for i in range(factor - 1):
        df_copy = df_final.copy()
        df_copy['augmentation_factor'] = i + 1
        dfs_augment.append(df_copy)
    df_final = pd.concat(dfs_augment, ignore_index=True)
    if len(df_final) > target_size:
        df_final = df_final.sample(n=target_size, random_state=42).reset_index(drop=True)
    print(f"OK Apres factorisation: {df_final.shape[0]} lignes")

print(f"\nDATASET AUGMENTE - TAILLE FINALE: {df_final.shape[0]} LIGNES, {df_final.shape[1]} COLONNES")

print("\n" + "=" * 80)
print("PHASE 7 : ENCODAGE ET NORMALISATION")
print("=" * 80)

numeric_cols = df_final.select_dtypes(include=[np.number]).columns.tolist()
text_cols = df_final.select_dtypes(include=['object']).columns.tolist()
print(f"Colonnes numeriques: {len(numeric_cols)}, texte: {len(text_cols)}")

print("\n1. Encodage des variables categoriques...")
cat_cols_to_encode = [c for c in ['bord_politique', 'niveau_geo', 'periode'] if c in text_cols]
if cat_cols_to_encode:
    df_final_encoded = pd.get_dummies(df_final, columns=cat_cols_to_encode, prefix=cat_cols_to_encode, drop_first=True)
    print(f"OK Variables encodees: {cat_cols_to_encode}")
else:
    df_final_encoded = df_final.copy()
    print("   Pas de variables categoriques a encoder")

print("\n2. Normalisation Min-Max...")
numeric_cols_final = df_final_encoded.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols_to_scale = [c for c in numeric_cols_final if df_final_encoded[c].nunique() > 2]
scaler = MinMaxScaler()
df_final_encoded[numeric_cols_to_scale] = scaler.fit_transform(df_final_encoded[numeric_cols_to_scale])
print(f"OK {len(numeric_cols_to_scale)} colonnes normalisees")

print("\n3. Verification des valeurs manquantes finales...")
missing_count = df_final_encoded.isnull().sum().sum()
if missing_count > 0:
    df_final_encoded = df_final_encoded.fillna(df_final_encoded.mean(numeric_only=True))
    for col in df_final_encoded.select_dtypes(include=['object']).columns:
        df_final_encoded[col] = df_final_encoded[col].fillna('UNKNOWN')
    print(f"OK {missing_count} valeurs manquantes traitees")
else:
    print("OK Aucune valeur manquante")

print(f"\nDATASET FINAL: {df_final_encoded.shape[0]} lignes, {df_final_encoded.shape[1]} colonnes")

print("\n" + "=" * 80)
print("PHASE 8 : SAUVEGARDE ET RAPPORT FINAL")
print("=" * 80)

print("\nSauvegarde CSV...")
df_final_encoded.to_csv(export_final_path, index=False, encoding='utf-8')
csv_size_mb = os.path.getsize(export_final_path) / (1024 * 1024)
print(f"OK CSV: {export_final_path} ({csv_size_mb:.2f} MB)")

parquet_path = export_final_path.replace('.csv', '.parquet')
print(f"Sauvegarde PARQUET...")
df_final_encoded.to_parquet(parquet_path, engine='pyarrow')
parquet_size_mb = os.path.getsize(parquet_path) / (1024 * 1024)
print(f"OK PARQUET: {parquet_path} ({parquet_size_mb:.2f} MB)")

print(f"\nSauvegarde finale dans NoSQL ({nosql_type})...")
if nosql_type is not None:
    records_final = df_final_encoded.to_dict(orient='records')
    if nosql_type == 'mongodb':
        final_col.delete_many({})
        if records_final: final_col.insert_many(records_final)
    elif nosql_type == 'tinydb':
        final_col.truncate()
        if records_final: final_col.insert_multiple(records_final)
    print("OK Donnees finales sauvegardees dans NoSQL.")

print("\nGeneration des visualisations...")
output_dir = os.path.join(base_path, 'outputs')
os.makedirs(output_dir, exist_ok=True)

plt.figure(figsize=(10, 6))
sns.countplot(data=df_final, x='orientation', palette='viridis')
plt.title("Distribution de l\'orientation politique")
plt.savefig(os.path.join(output_dir, "distribution_orientation.png"))
plt.close()

delta_pop_col = next((c for c in df_final.columns if 'delta' in c and 'pop' in c.lower()), None)
delta_emp_col = next((c for c in df_final.columns if 'delta' in c and 'emp' in c.lower()), None)
if delta_pop_col and delta_emp_col:
    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=df_final.sample(min(1000, len(df_final))),
                    x=delta_pop_col, y=delta_emp_col, hue='orientation', alpha=0.6)
    plt.title(f"Delta Population vs Delta Emploi")
    plt.savefig(os.path.join(output_dir, "scatter_delta_pop_emploi.png"))
    plt.close()

print("OK Visualisations sauvegardees dans outputs/")

print(f"\nSauvegarde des deltas intermediaires...")
if df_rev_delta is not None:
    df_rev_delta.to_parquet(os.path.join(delta_dir, 'delta_revenus.parquet'))
    print(f"   OK delta_revenus.parquet")
if df_emp_delta is not None:
    df_emp_delta.to_parquet(os.path.join(delta_dir, 'delta_emploi.parquet'))
    print(f"   OK delta_emploi.parquet")
if df_pop_delta is not None:
    df_pop_delta.to_parquet(os.path.join(delta_dir, 'delta_population.parquet'))
    print(f"   OK delta_population.parquet")

print("\n" + "=" * 80)
print("RAPPORT DE QUALITE FINAL")
print("=" * 80)

numeric_final = df_final_encoded.select_dtypes(include=[np.number]).shape[1]
categorical_final = df_final_encoded.select_dtypes(include=['object']).shape[1]
print(f"\nSTATISTIQUES GLOBALES:")
print(f"   Lignes: {df_final_encoded.shape[0]:,}")
print(f"   Colonnes: {df_final_encoded.shape[1]}")
print(f"   Colonnes numeriques: {numeric_final} ({100*numeric_final/(numeric_final+categorical_final+1e-9):.1f}%)")
print(f"   Colonnes categoriques: {categorical_final}")
if df_final_encoded.shape[0] >= 20000:
    print(f"   Taille minimale 20k: {df_final_encoded.shape[0]:,} OK")
else:
    print(f"   Taille: {df_final_encoded.shape[0]:,} < 20 000")

print(f"\nColonnes cles presentes:")
key_cols = ['CODGEO', 'Code_departement', 'annee', 'periode', 'est_perturbation', 'orientation']
print(f"   {[c for c in key_cols if c in df_final_encoded.columns]}")

print("\n" + "=" * 80)
print("PHASE 9 : ANALYSE DESCRIPTIVE")
print("=" * 80)

numeric_cols_final_rep = df_final_encoded.select_dtypes(include=[np.number]).columns.tolist()
if numeric_cols_final_rep:
    print(f"\nStatistiques descriptives (5 premieres colonnes numeriques):")
    print(df_final_encoded[numeric_cols_final_rep[:5]].describe().T.round(4))

if 'annee' in df_final_encoded.columns:
    print("\nDistribution par annee:")
    print(df_final_encoded['annee'].value_counts().sort_index())

if 'orientation' in df_final_encoded.columns:
    print("\nDistribution par orientation:")
    print(df_final_encoded['orientation'].value_counts())

print("\n" + "=" * 80)
print("OK PREPARATION DES DONNEES TERMINEE AVEC SUCCES")
print("=" * 80)
print(f"\nFichier principal: {export_final_path}")
print(f"Format PARQUET:    {parquet_path}")
